# Exercise 3 — Grand Finale Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shared.data_utils import load_dataset
rng = np.random.default_rng(0)

df = load_dataset('stock-prices')
PRICE = df.columns[1]   # second column = closing price
DATE = df.columns[0]
mask = rng.random(len(df)) < 0.05
df.loc[mask, PRICE] = np.nan
print('shape:', df.shape, ' price col:', PRICE)


**Step 1.** Inspect.

In [ ]:
print('NaNs in price:', df[PRICE].isna().sum())
print('date dtype  :', df[DATE].dtype)


**Step 2.** Parse + sort.

In [ ]:
df[DATE] = pd.to_datetime(df[DATE])
df = df.sort_values(DATE).reset_index(drop=True)
df.head()


**Step 3.** Forward-fill.

In [ ]:
df[PRICE] = df[PRICE].ffill()
assert df[PRICE].isna().sum() == 0


**Step 4.** 7-day moving average with NumPy.

In [ ]:
window = 7
kernel = np.ones(window) / window
ma = np.convolve(df[PRICE].values, kernel, mode='valid')
# Pad the front so lengths line up for plotting
ma_full = np.concatenate([np.full(window - 1, np.nan), ma])
print('ma len:', len(ma_full), '  df len:', len(df))


**Step 5.** Plot.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df[DATE], df[PRICE], label='price', alpha=0.5)
ax.plot(df[DATE], ma_full, label='7-day MA', lw=2)
ax.set_xlabel('date'); ax.set_ylabel(PRICE); ax.set_title('AAPL closing price + 7-day MA')
ax.legend(); fig.autofmt_xdate();
